In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import textwrap

# LOAD DATA
df = pd.read_csv("questionnaire_data_other_factors.csv")


def merge_groups(g):
    if g["n"].sum() == 0:
        return pd.Series({"association": 0, "n": 0})

    return pd.Series({
        "association": np.average(
            g["association"],
            weights=g["n"]
        ),
        "n": g["n"].sum()
    })


# Aggregate visits if multiple columns were merged
num_bin = df[df["type"].isin(["numeric", "binary"])].copy()

num_bin = (
    num_bin
    .groupby(
        ["score", "factor", "type"],
        as_index=False
    )
    .apply(
        merge_groups,
        include_groups=False
    )
    .reset_index()
)


cat = df[df["type"] == "categorical"].copy()

cat = (
    cat
    .groupby(
        ["score", "factor", "value"],
        as_index=False
    )
    .apply(
        merge_groups,
        include_groups=False
    )
    .reset_index()
)

cat["type"] = "categorical"


df = pd.concat(
    [num_bin, cat],
    ignore_index=True
)


OUT_DIR = Path("questionnaire_diagrams")
OUT_DIR.mkdir(exist_ok=True)


for questionnaire in sorted(df["score"].unique()):

    print(f"Processing {questionnaire}")

    qdf = df[df["score"] == questionnaire]


    for ftype in [
        "numeric",
        "binary",
        "categorical"
    ]:

        subset = qdf[qdf["type"] == ftype].copy()

        if subset.empty:
            continue


        # Clean invalid values
        subset["association"] = pd.to_numeric(
            subset["association"],
            errors="coerce"
        )

        subset = subset.replace(
            [np.inf, -np.inf],
            np.nan
        )

        subset = subset.dropna(
            subset=["association"]
        )

        if subset.empty:
            continue



        # Create labels
        if ftype == "categorical":

            subset["label"] = [
                textwrap.fill(
                    f"{row['factor']} = {row['value']}",
                    width=40
                )
                for _, row in subset.iterrows()
            ]

            subset = subset.sort_values(
                [
                    "factor",
                    "association"
                ]
            )

        else:

            subset = subset.sort_values(
                "association"
            )

            subset["label"] = [
                textwrap.fill(
                    str(label),
                    width=40
                )
                for label in subset["factor"]
            ]



        # Color mapping
        unique_factors = subset["factor"].unique()

        colors_cycle = (
            plt.rcParams["axes.prop_cycle"]
            .by_key()["color"]
        )

        color_map = {
            factor: colors_cycle[i % len(colors_cycle)]
            for i, factor in enumerate(unique_factors)
        }

        colors = [
            color_map[factor]
            for factor in subset["factor"]
        ]



        # Axis padding
        max_abs = subset["association"].abs().max()

        if pd.isna(max_abs) or max_abs == 0:
            max_abs = 1

        padding = max_abs * 0.3
        offset = max_abs * 0.03



        # Plot
        plt.figure(
            figsize=(
                14,
                max(8, len(subset) * 0.6)
            ),
            constrained_layout=True
        )


        bars = plt.barh(
            subset["label"],
            subset["association"],
            color=colors,
            alpha=0.7
        )


        plt.axvline(
            0,
            color="black",
            linewidth=1
        )


        # Extra room for text
        plt.xlim(
            subset["association"].min() - padding,
            subset["association"].max() + padding
        )


        # Add values
        for i, bar in enumerate(bars):

            value = subset["association"].iloc[i]
            n = int(subset["n"].iloc[i])


            if value >= 0:

                x_pos = value + offset
                alignment = "left"

            else:

                x_pos = value - offset
                alignment = "right"


            plt.text(
                x_pos,
                bar.get_y() + bar.get_height() / 2,
                f"{value:.3f} (n={n})",
                va="center",
                ha=alignment,
                fontsize=10,
                fontweight=(
                    "bold"
                    if abs(value) > 0.2
                    else "normal"
                ),
                clip_on=False
            )



        plt.title(
            f"{questionnaire.replace('_', ' ').title()} - "
            f"{ftype.capitalize()} Factors",
            fontsize=18,
            pad=20
        )


        plt.xlabel(
            "Spearman Correlation"
            if ftype == "numeric"
            else "Association (Mean Diff from global avg)",
            fontsize=14
        )


        plt.grid(
            axis="x",
            alpha=0.2
        )


        # Space for long labels
        plt.subplots_adjust(
            left=0.35
        )


        plt.savefig(
            OUT_DIR / f"{questionnaire}_{ftype}.png",
            dpi=300,
            bbox_inches="tight"
        )


        plt.close()



print(f"\nDONE. Saved to: {OUT_DIR}")

Processing chiq_result


C:\Users\karlw\AppData\Local\Temp\ipykernel_31120\1491909132.py:263: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(
C:\Users\karlw\AppData\Local\Temp\ipykernel_31120\1491909132.py:263: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(


Processing dass_depression


C:\Users\karlw\AppData\Local\Temp\ipykernel_31120\1491909132.py:263: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(
C:\Users\karlw\AppData\Local\Temp\ipykernel_31120\1491909132.py:263: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(


Processing dass_fear


...

Processing midas_result


C:\Users\karlw\AppData\Local\Temp\ipykernel_31120\1491909132.py:263: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(
C:\Users\karlw\AppData\Local\Temp\ipykernel_31120\1491909132.py:263: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(


Processing pgic_result


C:\Users\karlw\AppData\Local\Temp\ipykernel_31120\1491909132.py:263: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(
C:\Users\karlw\AppData\Local\Temp\ipykernel_31120\1491909132.py:263: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(



DONE. Saved to: questionnaire_diagrams
